# Architecture Patternsfor AI Apps

**Module 12 · Lesson 12.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The API Gateway: One Front Door
- The Model Router: The Right Model per Request
- Provider Fallback: Never Go Down
- Queue-Based Processing: Decouple Intake from Work
- Streaming: Time-to-First-Token
- The Tier Decision: Monolith, Microservices, or Serverless

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The demo that breaks on Monday

> 💡 **Info**
>
> Your notebook demo works: `model.generate("hi")` returns a friendly reply, and you ship it. Monday, real users arrive, and four things break at once. One request costs a rupee because “what are your hours?” went to the **frontier model**. Your provider has a 20-minute **outage** and every user sees a 500. A batch job holds an HTTP connection open for four minutes and **times out**. And the chat UI shows a **spinner for three seconds** before the first word appears, so users think it is broken. Not one of these is a model problem - they are all **architecture** problems. The model is the furniture; the gateway, the router, the fallback, the queue, and the stream are the building it lives in. This lesson, the opener for all of Module 12, builds that building - and you can run almost every piece of it with no key, because architecture is engineering logic.

### 🎯 What you will be able to do after this lesson

- **Build** the five core production patterns - an API gateway, a model router, a provider fallback, a job queue, and an SSE stream - as runnable services.

- **Compare** the deployment tiers (modular monolith vs microservices vs serverless) and know when to decompose.

- **Operate** each pattern’s mechanism: rate limiting, complexity routing, priority failover, submit-poll, and time-to-first-token streaming.

- **Evaluate** a whole reference architecture and map which Module 12 lesson deepens each layer.

> 📦 **Info**
>
> ✅ Before you startThis is the **Module 12 opener**. It builds on **11.9** (the async-job pattern - which becomes the queue here), your **Module 7** LLM-API and streaming basics (now behind a gateway), and **11.11** (the ship gate - this module is HOW you ship). Each pattern here gets its own deep lesson later in Module 12: reliability (12.2), gateways (12.3), caching (12.4), containers (12.5), CI/CD (12.6), scaling (12.7), monitoring (12.8).

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🏗️ **Analogy**
>
> A house is not a pile of bricks - it needs **architecture**: a front door with a lock (the gateway), hallways that send you to the right room (the router), a backup generator for when the power cuts (the fallback), a mailroom that takes your parcel and lets you leave (the queue), and an intercom that talks as you listen (the stream). The **model is the furniture**; the architecture is the building. **Where it breaks down:** a house is built once and stands; a production system is rebuilt continuously as traffic, providers, and models change - so these patterns are living infrastructure, not a one-time pour.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“`model.generate()` is the app.”** That one call is the furniture. The gateway, router, fallback, queue, and stream are the building - without them there is no production system, just a demo that breaks on Monday.
> - **“Send every query to the best model.”** Most traffic is simple. Tiered routing sends the bulk to a cheap model and reserves the frontier for hard queries, saving 80-90 percent with almost no quality loss.

> 💡 **Info**
>
> ⚠️ Anti-patternStarting with **microservices** before you need them, and **buffering** the whole LLM reply before returning it. In-process calls are nanoseconds while inter-service calls cost milliseconds plus operational overhead, so premature decomposition just adds latency and pain; and buffering makes a fast model feel slow. Start with a modular monolith, and stream tokens over SSE so time-to-first-token drives the experience.

---

## 🎯 Concept 1: The API Gateway: One Front Door

### The API Gateway: One Front Door

Every request enters through one layer that authenticates, rate-limits, logs, and routes. Cross-cutting concerns live in one place instead of on every endpoint.

Start with the foundation of every production AI app: the **API gateway**. Instead of scattering authentication, rate limiting, and logging across every endpoint, you put them in **one layer that every request passes through first**. The gateway **authenticates** the caller (an API key, a JWT), enforces a **rate limit** (so one client cannot exhaust your provider budget), **logs** every request for observability, and **routes** to the right AI service (`/chat`, `/embed`, `/image`). In an AI app it also centralizes your provider keys and cost tracking. The block runs a gateway over a batch of requests, keyless and deterministic (a tick counter stands in for the clock).

> 🚪 **Analogy**
>
> It is the **single front door of a building with a guard**. Everyone - residents, couriers, visitors - comes through the one door, where the guard checks IDs (auth), stops a crowd from rushing in all at once (rate limiting), notes who came and went (logging), and points each person to the right floor (routing). You do not put a guard on every internal room; you put one at the entrance. That is the gateway.

A client sends 4 requests in quick succession, and the gateway allows 3 per window. What happens to the 4th?

**📝 `01_gateway.py`** - *runnable*

In [ ]:
# THE API GATEWAY - one front door for every AI feature. Auth, rate limiting, logging, and
# routing all live in ONE layer instead of being re-implemented on every endpoint. We drive
# a batch of requests through it deterministically (a tick clock stands in for real time). No key.

API_KEYS = {"key-acme": "acme-corp"}      # valid keys -> tenant
RATE_LIMIT, WINDOW = 3, 10                 # 3 requests per 10 ticks, per key
seen = {}                                   # key -> list of request ticks (sliding window)
ROUTES = {"/v1/chat": "chat-service", "/v1/embed": "embedding-service", "/v1/image": "imagen-service"}

def gateway(tick, api_key, path):
    if api_key not in API_KEYS:
        return 401, "invalid API key"                      # AUTH
    hits = [t for t in seen.get(api_key, []) if tick - t < WINDOW]  # RATE LIMIT (sliding window)
    if len(hits) >= RATE_LIMIT:
        return 429, "rate limit exceeded"
    hits.append(tick); seen[api_key] = hits
    if path not in ROUTES:
        return 404, "no route"
    return 200, f"routed -> {ROUTES[path]}"                 # ROUTE

requests = [
    (0, "key-acme", "/v1/chat"),
    (1, "key-acme", "/v1/embed"),
    (2, "key-acme", "/v1/chat"),
    (3, "key-acme", "/v1/chat"),    # 4th within the window -> 429
    (4, "key-bad",  "/v1/chat"),    # bad key -> 401
]
print("Every request passes through the gateway (auth -> rate limit -> route), and is LOGGED:")
for tick, key, path in requests:
    status, msg = gateway(tick, key, path)
    print(f"  [t={tick}] {key:9} {path:9} -> {status} {msg}")
print("\nOne layer handles auth, rate limiting, logging, and routing for ALL AI features.")
# Output:
# Every request passes through the gateway (auth -> rate limit -> route), and is LOGGED:
#   [t=0] key-acme  /v1/chat  -> 200 routed -> chat-service
#   [t=1] key-acme  /v1/embed -> 200 routed -> embedding-service
#   [t=2] key-acme  /v1/chat  -> 200 routed -> chat-service
#   [t=3] key-acme  /v1/chat  -> 429 rate limit exceeded
#   [t=4] key-bad   /v1/chat  -> 401 invalid API key
#
# One layer handles auth, rate limiting, logging, and routing for ALL AI features.

- Every request flows through `gateway()`: auth first, then the rate check, then routing - the order matters.
- The first three requests from `key-acme` pass and route to their services; each is logged with its tick.
- The **4th within the window returns 429** (the sliding window already holds 3), and the **bad key returns 401**.
- All four cross-cutting concerns - auth, rate limit, logging, routing - live in this one layer, not on every endpoint.

#### 💡 What just happened

⚡ What just happened?The API gateway is the single front door: it authenticates, rate-limits, logs, and routes every request in one layer. Tradeoff: it adds a small hop (a few milliseconds), but it means you implement auth and rate limiting once instead of on every endpoint, and you get one place to observe and control all AI traffic. Managed gateways (Kong, Apigee) and AI gateways (LiteLLM, Portkey) go deep in Lesson 12.3.

- Requests flow through one gateway: auth, a sliding-window rate limiter, logging, routing
- The 4th request in the window trips a 429; a bad key trips a 401

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Model Router: The Right Model per Request

### The Model Router: The Right Model per Request

Classify each query’s complexity and send simple ones to a fast, cheap model and hard ones to a frontier model. Most traffic is simple, so this is the highest-ROI cost cut.

The single highest-ROI optimization in a production AI app is the **model router**. Most production traffic is **simple** - greetings, factual lookups, short answers - and sending all of it to an expensive frontier model is pure waste. A router **classifies each query’s complexity** (with a cheap classifier model, a heuristic, or embeddings) and sends simple queries to a **fast, cheap tier** and hard ones (reasoning, analysis, multi-step, code) to a **frontier tier**. Because the bulk of traffic is simple, **tiered routing commonly saves 80-90 percent** versus all-frontier, with almost no quality loss. The block runs a keyless heuristic router over three queries and computes the saving; the deep routing algorithms (and managed routers) are Lesson 12.3.

> 🏥 **Analogy**
>
> It is a **hospital triage nurse**. Not every patient needs the senior surgeon: the nurse sends a scraped knee to a general clinician and a chest pain to the cardiologist. Routing every case to the top specialist would be ruinously expensive and would clog the one resource you most need for the hard cases. The router triages queries the same way - cheap model for the routine, frontier model for the genuinely hard.

Two of three queries are simple greetings and one is a deep comparison. Routing simple -> cheap, how does cost compare to all-frontier?

**📝 `02_router.py`** - *runnable*

In [ ]:
# THE MODEL ROUTER - the right model per request. Most traffic is simple; sending every
# query to a frontier model is wasteful. Classify complexity, route simple -> a fast/cheap tier
# and hard -> a frontier tier. A cheap heuristic classifier stands in for a real one. No key.
# Prices are ILLUSTRATIVE per-query blended costs.

FAST     = {"tier": "fast/cheap", "cost": 0.001}     # e.g. a Haiku/Flash/Nano-class model
FRONTIER = {"tier": "frontier",   "cost": 0.020}     # e.g. a Sonnet/Opus/Pro-class model
HARD_SIGNALS = ("compare", "analyze", "why", "design", "code", "step by step", "npv", "trade-off")

def classify(q):
    ql = q.lower()
    if len(q) > 60 or any(s in ql for s in HARD_SIGNALS):
        return "complex"
    return "simple"

def route(q):
    return FRONTIER if classify(q) == "complex" else FAST

queries = ["Hi there!", "What are your hours?",
           "Compare the ROI of this course vs a CS masters with a 5-year NPV"]
print("Route each query by classified complexity:")
router_cost = 0.0
for q in queries:
    m = route(q)
    router_cost += m["cost"]
    print(f"  [{classify(q):7}] {q[:44]:44} -> {m['tier']} (${m['cost']}/query)")

all_frontier = len(queries) * FRONTIER["cost"]
saving = (all_frontier - router_cost) / all_frontier
print(f"\nrouter cost ${router_cost:.3f} vs all-frontier ${all_frontier:.3f} -> {saving:.0%} cheaper on THIS mix.")
print("At production ratios (roughly 90% of traffic simple), tiered routing saves 80-90%.")
# Output:
# Route each query by classified complexity:
#   [simple ] Hi there!                                    -> fast/cheap ($0.001/query)
#   [simple ] What are your hours?                         -> fast/cheap ($0.001/query)
#   [complex] Compare the ROI of this course vs a CS maste -> frontier ($0.02/query)
#
# router cost $0.022 vs all-frontier $0.060 -> 63% cheaper on THIS mix.
# At production ratios (roughly 90% of traffic simple), tiered routing saves 80-90%.

- `classify()` is a cheap heuristic: long queries or ones with hard signals (compare, analyze, npv, code) are *complex*; the rest are *simple*.
- The two greetings route to the **fast/cheap tier**; the ROI comparison routes to the **frontier tier**.
- On this mix the router costs **$0.022 vs $0.060 all-frontier - 63 percent cheaper**.
- At real production ratios (roughly 90 percent simple), the saving reaches 80-90 percent; the prices here are illustrative.

#### 💡 What just happened

⚡ What just happened?The model router classifies complexity and sends simple queries to a cheap tier, hard ones to the frontier, saving 80-90 percent on realistic traffic. Tradeoff: the classifier adds a little latency and can misroute, so you tune the threshold and fall back to the frontier tier when unsure. Routing algorithms and managed routers (RouteLLM, LiteLLM) are Lesson 12.3.

- Queries classified simple or complex route to a fast/cheap or frontier tier
- A cost meter compares the router with sending everything to the frontier model

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Provider Fallback: Never Go Down

### Provider Fallback: Never Go Down

Try providers in priority order; the first success wins. Because any single provider has outages, a multi-provider chain gives zero user-visible downtime.

Every LLM provider has outages, rate limits, and timeouts. If your app calls only one, its bad day is your bad day. **Provider fallback** fixes this: you list providers in **priority order** and try them in turn - the **first success wins**. If the primary is down, the request quietly fails over to the next, and the user never sees an error. You track **per-provider success and failure stats** so you can see which provider is flaky. This pairs with a **circuit breaker** (stop hammering a provider that keeps failing) which you will build in Lesson 12.2. The block runs a fallback chain with the primary down and shows the request served by the next provider.

> 🔌 **Analogy**
>
> It is a **backup generator**. A hospital does not shut down when the grid fails - the moment the power cuts, the generator kicks in and the lights stay on. Patients in surgery never notice the switch. A fallback chain is that generator for your AI app: when the primary provider goes dark, the request flips to the backup so fast that users never see the outage.

Your provider chain is [gemini (priority 1, DOWN), openai (2, up), claude (3, up)]. Who serves the request?

**📝 `03_fallback.py`** - *runnable*

In [ ]:
# PROVIDER FALLBACK - never go down. Try providers in priority order; the first success wins.
# Because any single provider has outages, a multi-provider chain gives zero user-visible
# downtime. We SCRIPT which providers are up (a real call would try the network). No key.

class Fallback:
    def __init__(self, providers):
        self.providers = sorted(providers, key=lambda p: p["priority"])
        self.stats = {p["name"]: {"ok": 0, "fail": 0} for p in providers}

    def call(self, prompt):
        for p in self.providers:
            if p["up"]:
                self.stats[p["name"]]["ok"] += 1
                return {"text": f"{p['name']}: {prompt[:24]}", "via": p["name"]}
            self.stats[p["name"]]["fail"] += 1
            print(f"  [{p['name']}] DOWN -> trying next...")
        return {"text": "all providers failed", "via": None}

# gemini is down today; openai and claude are up.
chain = Fallback([
    {"name": "gemini", "up": False, "priority": 1},
    {"name": "openai", "up": True,  "priority": 2},
    {"name": "claude", "up": True,  "priority": 3},
])
print("Call with the primary (gemini) down:")
r = chain.call("What is the refund policy?")
print(f"  -> served by {r['via']}: {r['text']}")
print(f"  stats: {chain.stats}")
print("The user never saw an error. (A circuit breaker to stop hammering gemini is Lesson 12.2.)")
# Output:
# Call with the primary (gemini) down:
#   [gemini] DOWN -> trying next...
#   -> served by openai: openai: What is the refund polic
#   stats: {'gemini': {'ok': 0, 'fail': 1}, 'openai': {'ok': 1, 'fail': 0}, 'claude': {'ok': 0, 'fail': 0}}
# The user never saw an error. (A circuit breaker to stop hammering gemini is Lesson 12.2.)

- `call()` tries providers **sorted by priority**: gemini first, then openai, then claude.
- gemini is down, so it logs the failure and **falls through to openai**, which succeeds - the first success wins.
- The `stats` record gemini’s failure and openai’s success, so you can spot a flaky provider over time.
- The user got a clean answer; if *all* providers were down, the chain returns a single clear error instead of a crash.

#### 💡 What just happened

⚡ What just happened?Provider fallback tries providers in priority order and returns the first success, giving zero user-visible downtime. Tradeoff: you must normalize each provider’s API and keep multiple keys, and a naive chain will hammer a dead provider - so you pair it with a circuit breaker and retries with backoff, which is Lesson 12.2.

- A request tries providers in priority order; the primary is down
- It falls through to the next provider and succeeds; a stats panel; an all-down case

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Queue-Based Processing: Decouple Intake from Work

### Queue-Based Processing: Decouple Intake from Work

Accept the request instantly and return a job_id; let independently-scaled workers process it; let the client poll or receive a webhook. This absorbs traffic spikes.

Some AI work is slow - a long document, a batch of images, a multi-step agent run - and holding an HTTP connection open for minutes is fragile: it times out, ties up a server, and tells the user nothing. The fix is **queue-based processing**: the API **accepts the request instantly and returns a `job_id`** (in well under a second), enqueues the work, and lets a pool of **independently-scaled workers** process it. The client **polls a status endpoint** (or receives a webhook) until the job is `completed`. This decouples intake from work, so a traffic spike fills the queue instead of crashing the API, and you scale the workers separately from the front end. This is 11.9’s async-job pattern, now at the architecture layer. The block runs the submit-poll lifecycle keylessly.

> 🍽️ **Analogy**
>
> It is a **restaurant kitchen ticket rail**. When you order, the waiter does not stand frozen at your table until the dish is cooked - they clip your ticket to the rail (the queue) and immediately take the next order. The kitchen (the workers) pulls tickets and cooks in parallel, and your food arrives when it is ready. The ticket is the `job_id`; submit-and-poll is exactly a kitchen ticket rail for slow work.

A request will take 4 minutes to process. With the queue pattern, how long until the API responds to the client?

**📝 `04_queue.py`** - *runnable*

In [ ]:
# QUEUE-BASED PROCESSING - decouple intake from work. Accept the request INSTANTLY (return a
# job_id in well under a second), enqueue it, let independently-scaled workers process it, and
# let the client POLL for status. This absorbs traffic spikes. A tick clock stands in for time. No key.

jobs = {}                        # job_id -> {status, result}
def submit(prompt):              # the API: returns immediately
    job_id = f"job-{len(jobs)+1}"
    jobs[job_id] = {"status": "queued", "prompt": prompt}
    return job_id
def worker_tick(job_id, tick):   # a background worker advances the job
    j = jobs[job_id]
    if tick == 1: j["status"] = "processing"
    if tick == 4: j.update(status="completed", result=f"generated: {j['prompt'][:20]}")
def poll(job_id):                # the client status endpoint
    return jobs[job_id]["status"]

jid = submit("write a 500-word product description")   # returns at t=0, ~1 tick
print(f"submit -> {jid} returned at t=0 (the API never blocks on the work)")
for t in range(5):
    worker_tick(jid, t)
    if t in (0, 1, 4):
        print(f"  poll at t={t}: {poll(jid)}")
print("\nIntake took ~1 tick; the work took 4. They scale independently. (This is 11.9 at the arch layer.)")
# Output:
# submit -> job-1 returned at t=0 (the API never blocks on the work)
#   poll at t=0: queued
#   poll at t=1: processing
#   poll at t=4: completed
#
# Intake took ~1 tick; the work took 4. They scale independently. (This is 11.9 at the arch layer.)

- `submit()` creates the job as `queued` and returns a `job_id` at **t=0** - the API never blocks on the work.
- `worker_tick()` is a background worker advancing the job: `processing` at t=1, `completed` at t=4.
- `poll()` is the `GET /status/{id}` endpoint the client hits - it reads queued, then processing, then completed.
- Intake took ~1 tick; the work took 4. They scale independently, so a spike fills the queue instead of crashing the API.

#### 💡 What just happened

⚡ What just happened?Queue-based processing decouples intake from work: submit returns a job_id instantly, independently-scaled workers process asynchronously, and the client polls or gets a webhook. Tradeoff: the client must poll (or subscribe) instead of getting an instant answer, but the API never blocks or times out and it absorbs spikes. Managed queues: Pub/Sub, Cloud Tasks, Redis, SQS.

- Submit returns a job_id instantly while the work runs in the background
- The job moves queued -> processing -> completed on scaled workers; the client polls

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Streaming: Time-to-First-Token

### Streaming: Time-to-First-Token

Stream tokens over SSE as the model generates them, so the first word appears almost immediately. Perceived speed is driven by time-to-first-token, not total latency.

An LLM generates tokens **sequentially over several seconds**. If you buffer the whole reply and return it at once, the user stares at a spinner and a perfectly fast model feels broken. **Streaming** fixes the *perceived* speed: you send each token **as it is produced**, so the first word appears almost immediately. The metric that matters is **time-to-first-token**, not total latency. The standard transport is **Server-Sent Events (SSE)**: it runs over plain HTTP, works with CDNs and reverse proxies, and gives the browser **auto-reconnect** for free (`EventSource`). You only need a **WebSocket** when the client must send messages *mid-stream* (cancel, tool approval, voice) - common in agent UIs, but not the default for chat. The block compares SSE streaming with batch delivery, keylessly.

> 📰 **Analogy**
>
> It is a **ticker tape versus waiting for the whole letter**. A ticker prints each word the instant it arrives, so you are reading the news within a second - that is SSE streaming. Buffering is waiting for the entire letter to be typed, sealed, and delivered before you can read a single word. Both take the same total time to finish, but the ticker *feels* immediate because you see the first words at once.

A reply is 9 tokens long. With SSE the first token lands at t=1; with batch delivery, when does the user first see anything?

**📝 `05_streaming.py`** - *runnable*

In [ ]:
# STREAMING - time-to-first-token, not total latency, drives the experience. An LLM generates
# tokens sequentially over seconds; buffering the whole reply feels slow. SSE streams each token
# as it is produced, so the first word lands almost immediately. A tick clock = time. No key.

reply = "Here is the answer to your question about refunds".split()
N = len(reply)

# STREAMING (SSE): emit token i at tick i+1; the user sees the first word at t=1.
print("STREAMING (SSE) - the user sees words appear:")
first_visible_stream = None
for i, tok in enumerate(reply):
    t = i + 1
    if first_visible_stream is None:
        first_visible_stream = t
print(f"  time-to-first-token = t={first_visible_stream}; full reply done at t={N}")

# BATCH: nothing is visible until the whole reply is buffered at t=N.
first_visible_batch = N
print("BATCH (buffer then return) - the user stares at a spinner:")
print(f"  first-visible = t={first_visible_batch} (the WHOLE reply at once)")

print(f"\nSSE shows the first word {first_visible_batch - first_visible_stream} ticks sooner "
      f"(t={first_visible_stream} vs t={first_visible_batch}). Same total time, far better perceived speed.")
print("SSE is the default for token streams (HTTP, CDN-friendly, auto-reconnect); WebSocket only for two-way.")
# Output:
# STREAMING (SSE) - the user sees words appear:
#   time-to-first-token = t=1; full reply done at t=9
# BATCH (buffer then return) - the user stares at a spinner:
#   first-visible = t=9 (the WHOLE reply at once)
#
# SSE shows the first word 8 ticks sooner (t=1 vs t=9). Same total time, far better perceived speed.
# SSE is the default for token streams (HTTP, CDN-friendly, auto-reconnect); WebSocket only for two-way.

- The reply is 9 tokens. With **SSE**, token *i* is emitted at tick *i+1*, so **time-to-first-token = t=1**.
- With **batch**, nothing is visible until the whole reply is buffered and returned at **t=9**.
- SSE shows the first word **8 ticks sooner** (t=1 vs t=9) - same total time, far better perceived speed.
- SSE is the default for token streams (HTTP, CDN-friendly, auto-reconnect); reach for a WebSocket only when the client sends mid-stream.

#### 💡 What just happened

⚡ What just happened?Streaming over SSE delivers tokens as they generate, so time-to-first-token, not total latency, drives perceived speed. Tradeoff: streaming complicates error handling mid-response and needs a proxy that does not buffer, but it is the difference between a fast-feeling chat and a broken-feeling spinner. Use a WebSocket only for genuinely two-way interaction.

- SSE streams tokens one by one; a time-to-first-token marker lands early
- A batch bar fills only at the end; same total time, very different feel

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The Tier Decision: Monolith, Microservices, or Serverless

### The Tier Decision: Monolith, Microservices, or Serverless

Start with a modular monolith because in-process calls are nearly free. Decompose to microservices only when something - usually GPU inference - must scale on its own. Serverless fits event-driven bursts.

Where do all these patterns *live*? Three deployment tiers, and the art is not over-engineering. Default to a **modular monolith**: one deployable with clear internal module boundaries, because **in-process function calls are nanoseconds** while inter-service calls cost **milliseconds plus operational overhead**. Amazon’s Prime Video famously moved one *audio/video quality-monitoring pipeline* off a microservices design (AWS Step Functions + S3) into a monolith and cut *that pipeline’s* cost roughly 90 percent; a CNCF 2025 report found around 42 percent of organizations consolidating some microservices back. Decompose to **microservices** only when a part must scale independently - for AI, the big one is **GPU inference** needing its own hardware and scaling - or when many teams need independent deploys. **Serverless** fits event-driven, bursty work: Cloud Run scales to zero (GPU cold start ~4-6s), dedicated serverless-GPU providers boot faster, and **AWS Lambda has no GPU** (orchestration only). The guiding discipline is **12-Factor Agents**: own your prompts, own your context window, and keep the agent a **stateless reducer** so it is easy to scale. The block routes app profiles to tiers.

> 🏠 **Analogy**
>
> It is **one big house versus a street of specialist shops**. For a family, one house is simplest: walk from kitchen to living room in a step (an in-process call). A street of separate shops (microservices) makes sense once each needs its own space, staff, and hours - but you would not build a whole shopping street to house one family, and walking between shops (inter-service calls) is slower than walking between rooms. You move to the street only when a shop genuinely needs to grow on its own - like the GPU-inference bakery that needs its own giant oven.

An API-calling AI app, 4 engineers, no self-hosted models. Which tier should it start on?

**📝 `06_tier_decision.py`** - *runnable*

In [ ]:
# THE TIER DECISION - modular monolith vs microservices vs serverless. Start with a modular
# monolith (in-process calls are nanoseconds); decompose to microservices only when something -
# usually GPU inference - must scale on its own; serverless fits event-driven, bursty work. No key.

def choose(self_hosted_gpu, independent_scaling, event_driven, big_team):
    if self_hosted_gpu and independent_scaling:
        return "MICROSERVICES", "GPU inference must scale on its own hardware"
    if big_team:
        return "MICROSERVICES", "many teams need independent deploys"
    if event_driven and not self_hosted_gpu:
        return "SERVERLESS", "event-driven + bursty -> scale to zero (Cloud Run / RunPod)"
    return "MODULAR MONOLITH", "start here: one deploy, in-process calls"

profiles = [
    ("API-calling MVP, 4 engineers",            dict(self_hosted_gpu=False, independent_scaling=False, event_driven=False, big_team=False)),
    ("self-hosted 70B model, own scaling",      dict(self_hosted_gpu=True,  independent_scaling=True,  event_driven=False, big_team=False)),
    ("nightly webhook + batch pipeline",        dict(self_hosted_gpu=False, independent_scaling=False, event_driven=True,  big_team=False)),
    ("30-engineer multi-team platform",         dict(self_hosted_gpu=False, independent_scaling=False, event_driven=False, big_team=True)),
]
print("Route each app profile to a deployment tier:")
for name, props in profiles:
    tier, why = choose(**props)
    print(f"  {name:38} -> {tier:16} ({why})")
print("\nDefault to a modular monolith; decompose only when you must. Lambda has NO GPU (orchestration only).")
# Output:
# Route each app profile to a deployment tier:
#   API-calling MVP, 4 engineers           -> MODULAR MONOLITH (start here: one deploy, in-process calls)
#   self-hosted 70B model, own scaling     -> MICROSERVICES    (GPU inference must scale on its own hardware)
#   nightly webhook + batch pipeline       -> SERVERLESS       (event-driven + bursty -> scale to zero (Cloud Run / RunPod))
#   30-engineer multi-team platform        -> MICROSERVICES    (many teams need independent deploys)
#
# Default to a modular monolith; decompose only when you must. Lambda has NO GPU (orchestration only).

- `choose()` defaults to a **modular monolith** and only decomposes on a real trigger.
- The API-calling MVP stays a **modular monolith**; the self-hosted 70B model with its own scaling goes to **microservices**.
- The nightly webhook + batch pipeline goes to **serverless** (event-driven, scale to zero); the 30-engineer platform goes to **microservices** for independent deploys.
- The rule: start monolith, decompose only when you must - and remember Lambda has no GPU, so serverless GPU means Cloud Run or a dedicated provider.

#### 💡 What just happened

⚡ What just happened?The tier decision: modular monolith by default, microservices when a part (usually GPU inference) must scale on its own or many teams need independent deploys, serverless for event-driven bursts. Tradeoff / the whole point: microservices buy independent scaling at the cost of latency and ops overhead, so you pay that price only when you have to. Scaling and serverless-GPU go deep in Lesson 12.7.

- Toggle app properties: self-hosted GPU? independent scaling? event-driven? big team?
- Watch it route to a modular monolith / microservices / serverless with the reason

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Reference Architecture: All Five, One Request

### The Reference Architecture: All Five, One Request

Trace a single request through gateway, router, cache, provider-with-fallback, and stream. Each pattern owns one concern; together they turn model.generate() into a production system.

Now assemble everything. A production request does not hit the model directly - it flows through the whole reference architecture, and each layer owns exactly one concern. It enters the **gateway** (auth, rate limit, log), goes to the **router** (pick the model tier), checks the **cache** (a hit returns instantly - Lesson 12.4), calls the **provider with fallback** (fail over if the primary is down - Lesson 12.2), and **streams** the result back token by token. Long jobs take a detour through the **queue** to independently-scaled workers. The block traces one chat request through every layer and shows where the time goes - ending at a time-to-first-token you can feel. This is the map of the whole module: each layer here gets its own deep lesson.

> 🏛️ **Analogy**
>
> It is the **finished building tour**. You walk in the front door past the guard (gateway), a concierge sends you to the right floor (router), you check whether your parcel is already at the desk (cache), a backup generator hums in case the power cuts (fallback), and an intercom narrates as you go (stream). Each part does one job, and only together do they make a building you can actually live and work in - not a pile of bricks.

In the traced request, which layer contributes the MOST latency, and why?

**📝 `07_reference_arch.py`** - *runnable*

In [ ]:
# THE REFERENCE ARCHITECTURE - all five patterns in ONE request. Trace a single chat request
# through every layer and watch where the time goes. Latencies are ILLUSTRATIVE milliseconds.
# This is the whole system: gateway -> router -> cache -> provider(fallback) -> stream. No key.

def trace_request(prompt):
    t = 0; log = []
    def step(layer, ms, note):
        nonlocal t; t += ms; log.append((layer, ms, t, note))
    step("gateway",  2, "auth ok, rate ok, logged")
    step("router",   1, "classified complex -> frontier tier")
    step("cache",    1, "miss (a hit would return here; see 12.4)")
    step("provider", 300, "gemini DOWN -> fell back to openai (see 12.2)")
    step("stream",   40, "first SSE token sent to the client (TTFT)")
    return log

print("One chat request, traced through the reference architecture:")
for layer, ms, cum, note in trace_request("what is the refund policy?"):
    print(f"  {layer:9} +{ms:>3}ms  (t={cum:>3}ms)  {note}")
print("\nTime-to-first-token = t=344ms. The gateway, router, fallback, and stream each own one concern;")
print("together they turn model.generate() into a production system. Each layer deepens in its own M12 lesson.")
# Output:
# One chat request, traced through the reference architecture:
#   gateway   +  2ms  (t=  2ms)  auth ok, rate ok, logged
#   router    +  1ms  (t=  3ms)  classified complex -> frontier tier
#   cache     +  1ms  (t=  4ms)  miss (a hit would return here; see 12.4)
#   provider  +300ms  (t=304ms)  gemini DOWN -> fell back to openai (see 12.2)
#   stream    + 40ms  (t=344ms)  first SSE token sent to the client (TTFT)
#
# Time-to-first-token = t=344ms. The gateway, router, fallback, and stream each own one concern;
# together they turn model.generate() into a production system. Each layer deepens in its own M12 lesson.

- One request is traced through every layer, with an illustrative per-layer latency and a running total.
- The **gateway, router, and cache checks are each only a few milliseconds** - nearly free.
- The **provider network call (here with a fallback hop) dominates** the total by far, as a real API round-trip always will.
- The first SSE token is sent at the end of that chain - the time-to-first-token the user actually feels - and each layer deepens in its own Module 12 lesson (reliability 12.2, gateways 12.3, caching 12.4, scaling 12.7).

**📝 `07b_fastapi_sse.py`** - *illustrative*

In [ ]:
%%javascript
# THE STANDARD AI API STACK - FastAPI + SSE (illustrative minimal example).
# FastAPI (async, Pydantic validation, OpenAPI docs) is the default Python framework for LLM
# services; a StreamingResponse with media_type "text/event-stream" is the standard token stream.
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import json

app = FastAPI(title="netsetos-ai-gateway")

async def token_stream(prompt: str):
    async for token in llm_client.stream(prompt):        # iterate the LLM's streaming response
        yield f"data: {json.dumps({'token': token})}\n\n"  # one SSE frame per token
    yield "data: [DONE]\n\n"

@app.get("/v1/stream")
async def stream(prompt: str = "Hello"):
    return StreamingResponse(token_stream(prompt), media_type="text/event-stream")

# Browser client:  const es = new EventSource("/v1/stream?prompt=hi"); es.onmessage = e => ...
# For a MULTI-PROVIDER gateway (fallback + routing across OpenAI / Anthropic / Google behind one
# OpenAI-compatible endpoint) you put LiteLLM or Portkey in front - that is Lesson 12.3.
# Output: (illustrative minimal example - needs `pip install fastapi uvicorn` + an ASGI server.)
# GET /v1/stream returns an SSE stream of {"token": ...} frames ending in [DONE]; EventSource reads it.

- `FastAPI` (async, Pydantic, OpenAPI) is the default Python framework for LLM services - the real version of the gateway and endpoints.
- `StreamingResponse` with `media_type="text/event-stream"` is the standard SSE token stream (Step 5), read by the browser’s `EventSource`.
- Each token is one `data: {...}` SSE frame; the stream ends with `[DONE]`.
- For a multi-provider gateway (fallback + routing behind one endpoint) you put **LiteLLM** or **Portkey** in front - that is Lesson 12.3.

#### 💡 What just happened

⚡ What just happened?The reference architecture is all five patterns in one request: gateway -> router -> cache -> provider(fallback) -> stream, with long jobs going through the queue. Tradeoff / the whole lesson: each layer adds a small hop, but together they turn a fragile model.generate() demo into a system that authenticates, controls cost, survives outages, absorbs spikes, and feels fast. That is what the rest of Module 12 makes production-grade.

- One request animates through gateway -> router -> cache -> provider(fallback) -> stream
- Each layer lights up with its latency; the provider call dominates; TTFT at the end

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture (and the map of Module 12)
> A production AI request is one journey through five patterns. The **gateway** guards the front door with auth, rate limiting, logging, and routing (Step 1). The **router** sends each query to the right model tier, saving 80-90 percent (Step 2). The **fallback** chain keeps you up when a provider goes down (Step 3). The **queue** decouples slow work from the request so spikes fill a buffer instead of crashing the API (Step 4). And **streaming** over SSE makes the first token land immediately, so the app feels fast (Step 5). All of it lives on a **modular monolith** until something - usually GPU inference - forces a decomposition (Step 6). Assembled, that is the **reference architecture** (Step 7): the building your model lives in. Every layer here gets its own deep lesson across the rest of Module 12.

| Deployment tier | Call cost | When to use | The AI trigger to decompose |
|---|---|---|---|
| Modular monolith | nanoseconds (in-process) | the default: API-calling apps, MVPs, small teams | - (start here) |
| Microservices | milliseconds + ops overhead | parts scale independently; many teams | GPU inference needs its own hardware/scaling |
| Serverless | per-request; cold starts | event-driven, bursty, scale-to-zero | webhooks / batch; Cloud Run for GPU (Lambda has none) |

> 📦 **Info**
>
> ➡️ Where this goes nextYou now have the map of a production AI system. Each layer deepens across the rest of Module 12: reliability and circuit breakers in Lesson 12.2, the AI gateway and routing landscape in Lesson 12.3, caching strategies in Lesson 12.4, containers in Lesson 12.5, CI/CD in Lesson 12.6, scaling and serverless-GPU in Lesson 12.7, and monitoring in Lesson 12.8. GPU economics and cost tradeoffs are covered in Module 13 (Cost & Performance); data-residency and compliance in Module 15 (Ethics & Responsible AI). The primary references are the [12-Factor Agents](https://github.com/humanlayer/12-factor-agents) guide, the [FastAPI SSE docs](https://fastapi.tiangolo.com/tutorial/server-sent-events/), the [Prime Video monolith case study](https://www.infoq.com/news/2023/05/prime-ec2-ecs-saves-costs/), and the [SSE vs WebSocket comparison](https://ably.com/blog/websockets-vs-sse).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Architecture Patternsfor AI Apps**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.1.html` - regenerate this notebook whenever the source HTML is updated.*
